# sktime-agentic-forecaster — demo

End-to-end walkthrough using the classic airline passengers dataset.
The agent picks the right sktime pipeline from a plain-English description.

In [ ]:
# Install if needed
# !pip install -e ..[anthropic]   # from repo root
# !pip install anthropic sktime

In [ ]:
import os
import matplotlib.pyplot as plt
from sktime.datasets import load_airline

from sktime_agent import AgenticForecaster

# Set your API key or export it as an env var before running
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["OPENAI_API_KEY"]    = "sk-..."

## 1. Load data

In [ ]:
y = load_airline()   # monthly airline passengers 1949–1960
y_train = y[:-12]    # hold out last 12 months for comparison
y_test  = y[-12:]

print(f"Training: {len(y_train)} obs  |  Test: {len(y_test)} obs")
y_train.plot(title="Airline passengers (training)");

## 2. Run the agentic forecaster

In [ ]:
forecaster = AgenticForecaster(llm_backend="anthropic", verbose=True)

result = forecaster.forecast(
    prompt="""
        Forecast the next 12 months of airline passenger numbers.
        The series has a clear upward trend and strong annual seasonality.
        Please use deseasonalisation and detrending before fitting.
    """,
    data=y_train,
    horizon=12,
)

## 3. Inspect the result

In [ ]:
print("Selected pipeline:", result.selected_estimators)
print("\nReasoning:")
print(result.explanation)
print("\nPredictions:")
print(result.predictions)

In [ ]:
# Inspect the actual sktime pipeline object
print(result.pipeline)

## 4. Plot predictions vs actuals

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
y_train.plot(ax=ax, label="Train", color="steelblue")
y_test.plot(ax=ax, label="Actual", color="grey", linestyle="--")
result.predictions.plot(ax=ax, label="Predicted", color="tomato")
ax.set_title(f"Airline passengers — {' → '.join(result.selected_estimators)}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Re-use the fitted pipeline

The returned `pipeline` is a real sktime estimator — you can refit it, serialize it, or use it in a cross-validation loop.

In [ ]:
from sktime.forecasting.base import ForecastingHorizon

# Refit on all data and forecast 6 more months
result.pipeline.fit(y)
extended = result.pipeline.predict(ForecastingHorizon(list(range(1, 7))))
print(extended)

## 6. Try a different LLM backend

Swap backends without changing any other code:

In [ ]:
# OpenAI
# result2 = AgenticForecaster(llm_backend="openai").forecast(
#     prompt="Monthly seasonal data, 12-month horizon, keep it simple",
#     data=y_train,
#     horizon=12,
# )

# LangChain (any compatible LLM)
# from langchain_openai import ChatOpenAI
# result3 = AgenticForecaster(llm_backend=ChatOpenAI(model="gpt-4o")).forecast(...)